# Van Genuchten SWRC

Este notebook muestra como ajustar los parámetros de Van Genuchten, propiedades como FC y PWP, y graficar

## Ecuación

$$\theta(h) = \theta_r + \dfrac{\theta_s - \theta_r}{\left[1 + (\alpha \, h)^n\right]^{1-1/n}}$$

| Símbolo | Descripción | Unidades |
|--------|-------------|-------|
| $\theta_s$ | Contenido de agua a saturación | % |
| $\theta_r$ | Contenido de agua residual | % |
| $\alpha$ | Parámetro de escala (relacionado con la entrada de aire) | cm⁻¹ |
| $n$ | Parámetro de forma | sin unidades |
| $h$ | Potencial mátrico = $10^{pF}$ | cm H₂O |

## Puntos hidráulicos de referencia

| Punto | pF | Potencial mátrico |
|-------|----|------------------|
| Capacidad de campo (FC) | 2.5 | −0.03 MPa |
| Punto de marchitez permanente (PWP) | 4.2 | −1.5 MPa |
| Agua disponible para plantas (PAW) | — | θ_FC − θ_PWP |

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D



## Cargar datos

In [ ]:
DATA_PATH = 'https://raw.githubusercontent.com/Saryace/swrc-functions/main/data/data-swrc.xlsx'

raw = pd.read_excel(DATA_PATH, engine='openpyxl')
raw.columns = ['pF', 'VWC']   # rename to short names
data = raw.dropna().copy()

data.head(8)

## Modelo Van Genuchten 

In [ ]:
def vg_unimodal(h, theta_s, theta_r, alpha, n):
    """
    Funcion VG.

    Parametros
    ----------
    h       : array-like, potencial mátrico (cm H₂O)
    theta_s : contenido agua saturacion (%)
    theta_r : contenido agua residual (%)
    alpha   : escala (cm⁻¹)
    n       : forma (>1)

    Returns
    -------
    theta   : contenido volumetrico (%)
    """
    m = 1.0 - 1.0 / n
    return theta_r + (theta_s - theta_r) / (1.0 + (alpha * h) ** n) ** m

In [ ]:
def fit_unimodal_vg(data: pd.DataFrame) -> dict:
    """
    Ajuste modelo van Genuchten.

    Parametros
    ----------
    data : DataFrame columnas 'pF' y 'VWC'

    Returns
    -------
    dict with keys: params, rsq, rmse, aic, model_type
    """
    df = data.copy()
    df['h'] = 10 ** df['pF']

    theta_s_init  = df['VWC'].max() * 0.95
    theta_r_init  = df['VWC'].min() * 1.05

    p0     = [theta_s_init, theta_r_init, 0.02, 1.5]
    bounds = (
        [theta_r_init, 0.0,           0.001, 1.1],   # lower
        [60.0,         theta_s_init,  1.0,   10.0],  # upper
    )

    popt, _ = curve_fit(
        vg_unimodal, df['h'], df['VWC'],
        p0=p0, bounds=bounds,
        method='trf', max_nfev=2000
    )

    theta_s, theta_r, alpha, n = popt

    fitted    = vg_unimodal(df['h'], *popt)
    residuals = df['VWC'].values - fitted
    ss_res    = np.sum(residuals ** 2)
    ss_tot    = np.sum((df['VWC'].values - df['VWC'].mean()) ** 2)

    rsq  = 1.0 - ss_res / ss_tot
    rmse = np.sqrt(np.mean(residuals ** 2))

    # AIC: 2k − 2·ln(L)  →  using Gaussian likelihood
    k   = 4
    n_obs = len(df)
    aic = n_obs * np.log(ss_res / n_obs) + 2 * k

    return {
        'params':     {'theta_s': theta_s, 'theta_r': theta_r,
                        'alpha': alpha, 'n': n},
        'rsq':        rsq,
        'rmse':       rmse,
        'aic':        aic,
        'model_type': 'unimodal'
    }

## FC, PWP, PAW

In [ ]:
def calculate_hydraulic(theta_s_uni_mean, theta_r_uni_mean,
                                  alpha_mean, n_mean) -> pd.DataFrame:
    """
    Calcula FC, PWP y PAW, 
    usando los parametros de VG.

    Referencias
    -----------------------
    FC  : pF = 2.5  → h = 316 cm  (≈ −0.03 MPa)
    PWP : pF = 4.2  → h = 15 849 cm  (≈ −1.5 MPa)
    """
    h_fc  = 10 ** 2.5
    h_pwp = 10 ** 4.2

    theta_fc  = vg_unimodal(h_fc,  theta_s_uni_mean, theta_r_uni_mean, alpha_mean, n_mean)
    theta_pwp = vg_unimodal(h_pwp, theta_s_uni_mean, theta_r_uni_mean, alpha_mean, n_mean)
    paw       = theta_fc - theta_pwp

    return pd.DataFrame({'theta_fc': [theta_fc],
                         'theta_pwp': [theta_pwp],
                         'paw': [paw]})

## Fit Curve

In [ ]:
fit = fit_unimodal_vg(data)
p   = fit['params']

print('── Fitted Parameters ─────────────────────────────')
for k, v in p.items():
    print(f'  {k:8s} = {v:.5f}')
print(f'  R²       = {fit["rsq"]:.4f}')
print(f'  RMSE     = {fit["rmse"]:.4f} %')
print(f'  AIC      = {fit["aic"]:.2f}')

In [ ]:
hydraulics = calculate_hydraulic(
    p['theta_s'], p['theta_r'], p['alpha'], p['n']
)

print('── Hydraulic Properties ──────────────────────────')
print(f'  Field Capacity   (pF 2.5)  θ_FC  = {hydraulics["theta_fc"].values[0]:.2f} %')
print(f'  Perm. Wilting Pt (pF 4.2)  θ_PWP = {hydraulics["theta_pwp"].values[0]:.2f} %')
print(f'  Plant Avail. Water         PAW   = {hydraulics["paw"].values[0]:.2f} %')

## Plot SWRC

In [ ]:
# ── fitted curve ─────────────────────────────────────────────────
pF_seq   = np.linspace(data['pF'].min(), data['pF'].max(), 500)
h_seq    = 10 ** pF_seq
vwc_fit  = vg_unimodal(h_seq, **p)

# Reference values
pF_fc   = 2.5
pF_pwp  = 4.2
vwc_fc  = hydraulics['theta_fc'].values[0]
vwc_pwp = hydraulics['theta_pwp'].values[0]
paw     = hydraulics['paw'].values[0]

# ── PAW shaded band ───────────────────────────────────────────────────────────
mask    = (pF_seq >= pF_fc) & (pF_seq <= pF_pwp)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6.5))

# Observed data
ax.scatter(data['pF'], data['VWC'],
           color='#5B8DB8', edgecolors='white', s=45, zorder=4,
           label='Observed data')

# Fitted curve
ax.plot(pF_seq, vwc_fit,
        color='#1A3A5C', lw=2, zorder=5,
        label='Unimodal VG fit')

# PAW shaded region
ax.fill_between(pF_seq[mask], vwc_pwp, vwc_fit[mask],
                color='#A8D5BA', alpha=0.45, label='PAW region')

# FC reference lines
ax.axvline(pF_fc,  linestyle='--', color='#2E8B57', lw=1.4)
ax.axhline(vwc_fc, linestyle='--', color='#2E8B57', lw=1.4)
ax.plot(pF_fc, vwc_fc, marker='D', ms=9,
        color='#2E8B57', markeredgecolor='white', zorder=6)
ax.annotate(
    f'FC\npF 2.5\nθ = {vwc_fc:.1f}%',
    xy=(pF_fc, vwc_fc),
    xytext=(pF_fc + 0.12, vwc_fc + 2.0),
    fontsize=9, color='#1A6B3A',
    arrowprops=dict(arrowstyle='->', color='#2E8B57', lw=1)
)

# PWP reference lines
ax.axvline(pF_pwp,  linestyle='--', color='#C0392B', lw=1.4)
ax.axhline(vwc_pwp, linestyle='--', color='#C0392B', lw=1.4)
ax.plot(pF_pwp, vwc_pwp, marker='D', ms=9,
        color='#C0392B', markeredgecolor='white', zorder=6)
ax.annotate(
    f'PWP\npF 4.2\nθ = {vwc_pwp:.1f}%',
    xy=(pF_pwp, vwc_pwp),
    xytext=(pF_pwp + 0.12, vwc_pwp + 2.0),
    fontsize=9, color='#8B1A1A',
    arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1)
)

# PAW text
ax.text((pF_fc + pF_pwp) / 2, (vwc_fc + vwc_pwp) / 2,
        f'PAW\n{paw:.1f}%',
        ha='center', va='center', fontsize=10,
        color='#1A6B3A', fontweight='bold')

# Legend
custom_legend = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#5B8DB8',
           markersize=8, label='Observed data'),
    Line2D([0],[0], color='#1A3A5C', lw=2, label='Unimodal VG fit'),
    mpatches.Patch(facecolor='#A8D5BA', alpha=0.6, label='PAW region'),
    Line2D([0],[0], color='#2E8B57', lw=1.4, ls='--',
           marker='D', ms=7, label='Field Capacity (FC)'),
    Line2D([0],[0], color='#C0392B', lw=1.4, ls='--',
           marker='D', ms=7, label='Perm. Wilting Point (PWP)'),
]
ax.legend(handles=custom_legend, loc='upper right', framealpha=0.9, fontsize=9)

ax.set_xlim(data['pF'].min() - 0.1, data['pF'].max() + 0.1)
ax.set_ylim(0, 52)
ax.set_xlabel('pF  [log₁₀(cm H₂O)]', fontsize=12)
ax.set_ylabel('Volumetric Water Content (%)', fontsize=12)
ax.set_title(
    'Soil Water Retention Curve — van Genuchten Model\n'
    'Field Capacity (FC), Permanent Wilting Point (PWP), and Plant Available Water (PAW)',
    fontsize=12, fontweight='bold'
)
ax.grid(True, which='major', alpha=0.3)
ax.set_xticks(range(0, 8))

plt.tight_layout()
plt.show()
